In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.collections import PatchCollection
import re

#File paths
INFILE = os.path.join("data", "2-output-data", "economic_summary_morbidity_sensitivity.csv")
OUTDIR = os.path.join("data", "2-output-data", "plots")
os.makedirs(OUTDIR, exist_ok=True)

In [2]:
# -------------------------
# Helpers
# -------------------------
_num_pat = re.compile(r"^\s*([0-9,\.]+)\s*\[\s*([0-9,\.]+)\s*-\s*([0-9,\.]+)\s*\]\s*$")
def parse_est_ci(s):
    """Parse '19,250 [13,700-24,800]' -> (central, lci, uci) floats."""
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return (np.nan, np.nan, np.nan)
    s = str(s).strip()
    if not s:
        return (np.nan, np.nan, np.nan)
    m = _num_pat.match(s)
    if not m:
        return (np.nan, np.nan, np.nan)
    c, l, u = m.groups()
    c = float(c.replace(",", ""))
    l = float(l.replace(",", ""))
    u = float(u.replace(",", ""))
    return (c, l, u)

def _to_float(x):
    if x is None:
        return np.nan
    if isinstance(x, str) and x.strip() == "":
        return np.nan
    try:
        return float(x)
    except Exception:
        return np.nan

def clean_scenario(sc):
    return str(sc).strip().lower()

def outcome_display_name(outcome):
    return OUTCOME_LABELS.get(outcome, outcome)

def is_mortality_row(disease):
    return str(disease).strip() == MORT_DISEASE_LABEL

def is_any_mortality_variant(disease):
    # excludes sensitivity rows too (if present)
    return str(disease).lower().startswith("all-cause mortality")

def safe_sum(series):
    # sum treating all-NaN as NaN (min_count=1) but allow 0 for empty after fill
    return series.sum(min_count=1)

def fmt_ci(c, l, u, digits=0):
    if c is None or (isinstance(c, float) and np.isnan(c)):
        return "--"
    if digits == 0:
        return f"{c:,.0f} [{l:,.0f}–{u:,.0f}]"
    return f"{c:,.{digits}f} [{l:,.{digits}f}–{u:,.{digits}f}]"

def fmt_millions_ci(c, l, u):
    if c is None or (isinstance(c, float) and np.isnan(c)):
        return "--"
    return f"{c:,.0f} [{l:,.0f}–{u:,.0f}]"
def get_multiplier(scenario, pollutant, year, disease):
    row = lag_df[
        (lag_df["Scenario"] == scenario)
        & (lag_df["Pollutant"] == pollutant)
        & (lag_df["Year"] == str(year))
        & (lag_df["Disease"] == disease)
        ]
    if not row.empty:
        return float(row.iloc[0]["lag_multiplier"])
    return 1.0

def morbidity_csv_path(scenario, pollutant, year):
    for sc_variant in (scenario, scenario.lower(), scenario.upper()):
        p = os.path.join(base_path, sc_variant, pollutant, str(year), "morbidity_results.csv")
        if os.path.exists(p):
            return p
    return None
def mortality_csv_path(scenario, pollutant, year):
    for sc_variant in (scenario, scenario.lower(), scenario.upper()):
        for fname in ("mortality_chimere_mc.csv", "mortality_chimere.csv"):
            p = os.path.join(base_path, sc_variant, pollutant, str(year), fname)
            if os.path.exists(p):
                return p
    return None

def norm_cols(df):
    df = df.copy()
    df.columns = [str(c).strip().replace("\ufeff", "") for c in df.columns]
    return df

def pick_col(df, candidates):
    lowmap = {str(c).strip().lower(): c for c in df.columns}
    for c in candidates:
        if str(c).lower() in lowmap:
            return lowmap[str(c).lower()]
    return None

def filter_morbidity_diseases(df):
    out = []
    for disease_key, (min_age, max_age) in disease_age_groups.items():
        mask = df["disease"].str.contains(disease_key, case=False, na=False) | (df["disease"] == disease_key)
        dfd = df[mask].copy()
        dfd = dfd[(dfd["age"] >= min_age) & (dfd["age"] <= max_age)]
        if not dfd.empty:
            dfd["disease"] = disease_key
            out.append(dfd)
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=df.columns)

def mortality_csv_path(scenario, pollutant, year):
    for sc_variant in (scenario, scenario.lower(), scenario.upper()):
        p = os.path.join(base_path, sc_variant, pollutant, str(year), "mortality_chimere.csv")
        if os.path.exists(p):
            return p
    return None

def morbidity_csv_path(scenario, pollutant, year):
    for sc_variant in (scenario, scenario.lower(), scenario.upper()):
        p = os.path.join(base_path, sc_variant, pollutant, str(year), "morbidity_results.csv")
        if os.path.exists(p):
            return p
    return None

def _norm_cols(df):
    df = df.copy()
    df.columns = [str(c).strip().replace("\ufeff", "") for c in df.columns]
    return df

def _pick_col(df, candidates):
    lowmap = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        hit = lowmap.get(cand.lower())
        if hit is not None:
            return hit
    return None

def load_total_ylg(mortality_csv):
    df = _norm_cols(pd.read_csv(mortality_csv))
    c_med = _pick_col(df, ["overall_YLG", "YLG", "overall_ylg"])
    c_lci = _pick_col(df, ["overall_YLG_LCI", "overall_YLG_lci", "YLG_LCI", "YLG_lci"])
    c_uci = _pick_col(df, ["overall_YLG_UCI", "overall_YLG_uci", "YLG_UCI", "YLG_uci"])
    if c_med is None:
        raise ValueError(f"Could not find YLG column in {mortality_csv}")

    med = pd.to_numeric(df[c_med], errors="coerce").dropna()
    lci = pd.to_numeric(df[c_lci], errors="coerce").dropna() if c_lci else pd.Series([np.nan])
    uci = pd.to_numeric(df[c_uci], errors="coerce").dropna() if c_uci else pd.Series([np.nan])

    return float(med.iloc[0]), float(lci.iloc[0]) if not lci.empty else np.nan, float(
        uci.iloc[0]) if not uci.empty else np.nan

def load_total_yld(morbidity_csv):
    df = _norm_cols(pd.read_csv(morbidity_csv))
    c_med = _pick_col(df, ["YLD_central", "YLD", "yld_central", "yld"])
    c_lci = _pick_col(df, ["YLD_low", "yld_low", "YLD_LCI", "yld_lci"])
    c_uci = _pick_col(df, ["YLD_high", "yld_high", "YLD_UCI", "yld_uci"])
    if c_med is None:
        raise ValueError(f"Could not find YLD column in {morbidity_csv}")

    med = pd.to_numeric(df[c_med], errors="coerce").fillna(0).sum()
    lci = pd.to_numeric(df[c_lci], errors="coerce").fillna(0).sum() if c_lci else np.nan
    uci = pd.to_numeric(df[c_uci], errors="coerce").fillna(0).sum() if c_uci else np.nan
    return float(med), float(lci), float(uci)

In [3]:
# Your main outcome table: only 'no_lag'
df = pd.read_csv(INFILE)
df = df[df["Lag"].astype(str).str.lower() == "cessation_lag"].copy()
if df.empty:
    raise RuntimeError("No rows found with Lag == 'cessation_lag'. Check your CSV.")
df["Scenario"] = df["Scenario"].apply(clean_scenario)
df["Year"] = df["Year"].astype(str)
df["Pollutant"] = df["Pollutant"].astype(str)
df["Disease"] = df["Disease"].astype(str)
df = df[df["Lag"].astype(str).str.lower() == "cessation_lag"].copy()
if df.empty:
    raise RuntimeError("No rows found with Lag == 'cessation_lag'. Check your CSV for correct rows.")

lag_multiplier_df = df.copy()
# Choose lag factor: morbidity or mortality
MORT_DISEASE_LABEL = "All-cause mortality (main)"
morb_mask = ~lag_multiplier_df["Disease"].eq(MORT_DISEASE_LABEL)
lag_multiplier_df["lag_multiplier"] = pd.Series(
    np.where(
        morb_mask,
        pd.to_numeric(lag_multiplier_df["Lag factor used (morbidity)"], errors="coerce"),
        pd.to_numeric(lag_multiplier_df["Lag factor used (mortality)"], errors="coerce"),
    ),
    index=lag_multiplier_df.index,
)
lag_multiplier_df["lag_multiplier"] = lag_multiplier_df["lag_multiplier"].fillna(1.0)
# lag_multiplier_df is now ready for merge

In [5]:
# --------------------------------------------------------
# Figure 1: Mortality avoided only
# Vertical bars with LCI/UCI and % gains relative to baseline mortality
# --------------------------------------------------------

base_dir = "data/2-output-data"

scenarios = ["s1", "s2", "s3", "s4"]
scenario_labels = {"s1": "S1", "s2": "S2", "s3": "S3", "s4": "S4"}
scenario_colors = {"S1": "#9695FF", "S2": "#B47CC7", "S3": "#6ABF69", "S4": "#E88D67"}
pollutant_labels = {
    "ug_PM25_RH50": "PM2.5",
    "ug_NO2": "NO2"
}
years = ["2030", "2050"]

MORT_DISEASE_LABEL = "All-cause mortality (main)"

BASELINE_MORTALITY = {
    "2030": 693893.5,
    "2050": 882564.6,
}

# --------------------------------------------------------
# BUILD DATAFRAME FROM SAVED FILES
# --------------------------------------------------------
records = []

for scenario in scenarios:
    for pollutant_key, pollutant_name in pollutant_labels.items():
        for year in years:
            filepath = os.path.join(base_dir, scenario, pollutant_key, year, "mortality_chimere.csv")

            if not os.path.exists(filepath):
                print(f"⚠ Missing file: {filepath}")
                continue

            df = pd.read_csv(filepath)
            df.columns = [str(col).strip().replace("\ufeff", "") for col in df.columns]
            normalized_col_map = {str(col).strip().lower(): col for col in df.columns}

            med_col = normalized_col_map.get("overall_avoided_deaths")
            lci_col = normalized_col_map.get("overall_avoided_deaths_lci")
            uci_col = normalized_col_map.get("overall_avoided_deaths_uci")

            if med_col is None or lci_col is None or uci_col is None:
                print(f"⚠ Missing mortality avoided columns in: {filepath}")
                continue

            median_val = pd.to_numeric(df[med_col], errors="coerce").dropna()
            lci_val = pd.to_numeric(df[lci_col], errors="coerce").dropna()
            uci_val = pd.to_numeric(df[uci_col], errors="coerce").dropna()

            if median_val.empty or lci_val.empty or uci_val.empty:
                continue

            records.append([
                year,
                scenario_labels[scenario],
                pollutant_name,
                float(median_val.iloc[0]),
                float(lci_val.iloc[0]),
                float(uci_val.iloc[0]),
            ])

df_plot = pd.DataFrame(
    records,
    columns=["Year", "Scenario", "Pollutant", "Median", "LCI", "UCI"]
)

# --------------------------------------------------------
# APPLY LAG MULTIPLIER TO MORTALITY AVOIDED
# --------------------------------------------------------
df_plot["Disease"] = MORT_DISEASE_LABEL

df_plot = df_plot.merge(
    lag_multiplier_df[["Scenario", "Pollutant", "Year", "Disease", "lag_multiplier"]],
    on=["Scenario", "Pollutant", "Year", "Disease"],
    how="left"
)
df_plot["lag_multiplier"] = df_plot["lag_multiplier"].fillna(1.0)

for col in ["Median", "LCI", "UCI"]:
    df_plot[col] = df_plot[col] * df_plot["lag_multiplier"]

df_plot["Baseline_mortality"] = df_plot["Year"].map(BASELINE_MORTALITY)
df_plot["Pct_gain"] = 100.0 * df_plot["Median"] / df_plot["Baseline_mortality"]
df_plot["Pct_gain_LCI"] = 100.0 * df_plot["LCI"] / df_plot["Baseline_mortality"]
df_plot["Pct_gain_UCI"] = 100.0 * df_plot["UCI"] / df_plot["Baseline_mortality"]

save_dir = os.path.join(base_dir, "plots")
os.makedirs(save_dir, exist_ok=True)

years_order = years
scenarios_to_plot = ["S1", "S2", "S3", "S4"]
pollutants = ["PM2.5", "NO2"]

bar_w = 0.17
x_base = np.arange(len(years_order), dtype=float)
scenario_offset = {
    "S1": -1.5 * bar_w,
    "S2": -0.5 * bar_w,
    "S3": 0.5 * bar_w,
    "S4": 1.5 * bar_w,
}

whisker_lw = 1.0
cap_size = 3.0
whisker_ls = "dotted"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
})

fig, axes = plt.subplots(1, len(pollutants), figsize=(11.8, 5.2), dpi=300, sharey=False)
axes = np.atleast_1d(axes)

for ax, pollutant in zip(axes, pollutants):
    df_pol = df_plot[df_plot["Pollutant"] == pollutant].copy()
    if df_pol.empty:
        ax.set_visible(False)
        continue

    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle="-", linewidth=0.45, alpha=0.22)
    ax.grid(axis="x", visible=False)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    max_uci = pd.to_numeric(df_pol["UCI"], errors="coerce").max()
    ymax = max_uci * 1.28 if np.isfinite(max_uci) and max_uci > 0 else 1.0

    for sc in scenarios_to_plot:
        sub = df_pol[df_pol["Scenario"] == sc].copy()

        med, lci, uci, pct = [], [], [], []
        for yr in years_order:
            row = sub[sub["Year"] == yr]
            if row.empty:
                med.append(np.nan)
                lci.append(np.nan)
                uci.append(np.nan)
                pct.append(np.nan)
            else:
                med.append(float(row["Median"].iloc[0]))
                lci.append(float(row["LCI"].iloc[0]))
                uci.append(float(row["UCI"].iloc[0]))
                pct.append(float(row["Pct_gain"].iloc[0]))

        med = np.array(med, dtype=float)
        lci = np.array(lci, dtype=float)
        uci = np.array(uci, dtype=float)
        pct = np.array(pct, dtype=float)
        xpos = x_base + scenario_offset[sc]

        ax.bar(
            xpos,
            med,
            width=bar_w,
            color=scenario_colors[sc],
            edgecolor="black",
            linewidth=0.55,
            alpha=0.94,
            label=sc,
            zorder=3
        )

        for x, lo, hi in zip(xpos, lci, uci):
            if np.isfinite(x) and np.isfinite(lo) and np.isfinite(hi):
                ax.vlines(
                    x,
                    lo,
                    hi,
                    colors="black",
                    linestyles=whisker_ls,
                    linewidth=whisker_lw,
                    zorder=4
                )
                ax.hlines(
                    [lo, hi],
                    x - cap_size * 0.006,
                    x + cap_size * 0.006,
                    colors="black",
                    linestyles=whisker_ls,
                    linewidth=whisker_lw,
                    zorder=4
                )

    ax.set_xticks(x_base)
    ax.set_xticklabels(years_order)
    ax.set_xlabel("Year")
    ax.set_ylabel("Avoided premature deaths (>30 years)")
    ax.set_title(f"Mortality avoided — {pollutant}", fontweight="bold", pad=10)
    ax.set_ylim(0, ymax)

    ax.yaxis.set_major_formatter(lambda x, pos: f"{x:,.0f}")

legend_items = [
    Patch(facecolor=scenario_colors["S1"], edgecolor="black", linewidth=0.55, label="S1"),
    Patch(facecolor=scenario_colors["S2"], edgecolor="black", linewidth=0.55, label="S2"),
    Patch(facecolor=scenario_colors["S3"], edgecolor="black", linewidth=0.55, label="S3"),
    Patch(facecolor=scenario_colors["S4"], edgecolor="black", linewidth=0.55, label="S4"),
]

fig.legend(
    handles=legend_items,
    title="Scenario",
    frameon=True,
    ncol=4,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    columnspacing=1.4,
    handlelength=1.2
)

fig.tight_layout(rect=[0, 0, 1, 0.92])
out = os.path.join(save_dir, "Figure2_PM25_NO2_matrix.png")
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"Saved plots to: {save_dir}")


Saved plots to: data/2-output-data\plots


In [12]:
# --------------------------------------------------------
# PLOTTING Figure 2- (3 panels: absolute, % attributable baseline, % total baseline)
# --------------------------------------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# SETTINGS
base_dir = "data/2-output-data"

scenarios = ["s1", "s2", "s3", "s4"]
scenario_labels = {"s1": "S1", "s2": "S2", "s3": "S3", "s4": "S4"}
scenario_colors = {"S1": "#9695FF", "S2": "#B47CC7", "S3": "#6ABF69", "S4": "#E88D67"}
pollutant_labels = {"ug_PM25_RH50": "PM2.5", "ug_NO2": "NO2"}
years = ["2030", "2050"]

MORT_DISEASE_LABEL = "All-cause mortality (main)"  # Must match lag_multiplier_df value
# --------------------------
# FIXED BASELINES (WHO-threshold excess burden, 2019 based on new HRAPIE-report)
# --------------------------
BASELINE_ATTR = {
    "PM2.5": {"median": 28160.0, "lci": 17400.0, "uci": 35900.0},
    "NO2": {"median": 30300.0, "lci": 18500.0, "uci": 41600.0},
}


# --------------------------------------------------------
# Helper to read avoided deaths from a given filename
# --------------------------------------------------------
def build_records_from_file(filename: str, parameter_name: str):
    """
    Reads scenario/pollutant/year files and returns records with columns:
    Year, Scenario, Pollutant, Parameter, Median, LCI, UCI
    """
    recs = []
    for scenario in scenarios:
        for pollutant_key, pollutant_name in pollutant_labels.items():
            for year in years:
                filepath = os.path.join(base_dir, scenario, pollutant_key, year, filename)

                if not os.path.exists(filepath):
                    print(f"⚠ Missing file: {filepath}")
                    continue

                df = pd.read_csv(filepath)
                df.columns = [str(col).strip().replace("\ufeff", "") for col in df.columns]
                normalized_col_map = {str(col).strip().lower(): col for col in df.columns}

                med_col = normalized_col_map.get("overall_avoided_deaths")
                lci_col = normalized_col_map.get("overall_avoided_deaths_lci")
                uci_col = normalized_col_map.get("overall_avoided_deaths_uci")

                if med_col is None or lci_col is None or uci_col is None:
                    print(f"⚠ Missing avoided death columns in: {filepath}")
                    continue

                median_val = pd.to_numeric(df[med_col], errors="coerce").dropna()
                lci_val = pd.to_numeric(df[lci_col], errors="coerce").dropna()
                uci_val = pd.to_numeric(df[uci_col], errors="coerce").dropna()

                if median_val.empty or lci_val.empty or uci_val.empty:
                    continue

                recs.append([
                    year,
                    scenario_labels[scenario],
                    pollutant_name,
                    parameter_name,
                    float(median_val.iloc[0]),
                    float(lci_val.iloc[0]),
                    float(uci_val.iloc[0]),
                ])
    return recs


# --------------------------------------------------------
# BUILD DATAFRAME(S)
#   - MAIN scenario values from mortality_chimere.csv (for panel 1 + panel 3)
#   - WHO-threshold run scenario values from mortality_chimere_WHO.csv (for panel 2)
# --------------------------------------------------------
main_records = build_records_from_file("mortality_chimere.csv", "MortalityAvoided_main")
who_records = build_records_from_file("mortality_chimere.csv", "MortalityWHO_main")

df_main = pd.DataFrame(
    main_records,
    columns=["Year", "Scenario", "Pollutant", "Parameter", "Median", "LCI", "UCI"]
)
df_who = pd.DataFrame(
    who_records,
    columns=["Year", "Scenario", "Pollutant", "Parameter", "Median", "LCI", "UCI"]
)

# Combine (we'll later derive plotted metrics and keep them in df_plot)
df_plot = pd.concat([df_main, df_who], ignore_index=True)

# --------------------------------------------------------
# APPLY LAG MULTIPLIER
# - Apply ONLY to MAIN avoided deaths (panel 1 + panel 3).
# - Do NOT apply to WHO-file scenario mortality used in panel 2.
# --------------------------------------------------------
df_plot["Disease"] = MORT_DISEASE_LABEL
df_plot = df_plot.merge(
    lag_multiplier_df[["Scenario", "Pollutant", "Year", "Disease", "lag_multiplier"]],
    on=["Scenario", "Pollutant", "Year", "Disease"],
    how="left"
)
df_plot["lag_multiplier"] = df_plot["lag_multiplier"].fillna(1.0)

mask_main_avoided = df_plot["Parameter"].eq("MortalityAvoided_main")
for col in ["Median", "LCI", "UCI"]:
    df_plot.loc[mask_main_avoided, col] = (
            df_plot.loc[mask_main_avoided, col] * df_plot.loc[mask_main_avoided, "lag_multiplier"]
    )
# --------------------------------------------------------
# DERIVE:
#  - Panel 1: absolute from MAIN file (lag-adjusted)
#  - Panel 2: % change vs baseline using WHO file values:
#       (BASELINE_ATTR - scenario_mort_from_WHO) / BASELINE_ATTR * 100
#  - Panel 3: % of total all-cause using MAIN file values (lag-adjusted)
# --------------------------------------------------------
derived_rows = []

# Panel 1: rename main parameter
main_abs = df_plot[df_plot["Parameter"] == "MortalityAvoided_main"].copy()
if not main_abs.empty:
    tmp = main_abs.copy()
    tmp["Parameter"] = "MortalityAvoided"
    derived_rows.append(tmp)

# Panel 2: % change vs baseline from "scenario mortality" values (NOT lag-adjusted)
who_abs = df_plot[df_plot["Parameter"] == "MortalityWHO_main"].copy()
for _, r in who_abs.iterrows():
    pol = r["Pollutant"]
    if pol not in BASELINE_ATTR:
        continue

    b = BASELINE_ATTR[pol]
    b_med = float(b["median"])
    b_lci = float(b["lci"])
    b_uci = float(b["uci"])

    # scenario mortality values from WHO file
    scen_med = float(r["Median"])
    scen_lci = float(r["LCI"])
    scen_uci = float(r["UCI"])

    # Relative % change vs baseline:
    # # remaining% = (scenario_attrib / baseline_attrib) * 100
    if b_med:
        pct_med = 100.0 * scen_med / b_med
        pct_lci = 100.0 * scen_lci / b_med
        pct_uci = 100.0 * scen_uci / b_med
    else:
        pct_med = np.nan
        pct_lci = np.nan
        pct_uci = np.nan

    derived_rows.append(pd.DataFrame([{
        "Year": r["Year"],
        "Scenario": r["Scenario"],
        "Pollutant": pol,
        "Parameter": "MortalityAvoided_pct_baseline",
        "Median": pct_med,
        "LCI": pct_lci,
        "UCI": pct_uci,
        "Disease": r["Disease"],
        "lag_multiplier": r["lag_multiplier"],
    }]))

# Panel 3: compute % total all-cause from MAIN file values (lag-adjusted)
for _, r in main_abs.iterrows():
    yr = r["Year"]
    denom = 583156
    if not np.isfinite(denom) or denom == 0:
        continue
    derived_rows.append(pd.DataFrame([{
        "Year": yr,
        "Scenario": r["Scenario"],
        "Pollutant": r["Pollutant"],
        "Parameter": "MortalityAvoided_pct_total",
        "Median": 100.0 * float(r["Median"]) / denom,
        "LCI": 100.0 * float(r["LCI"]) / denom,
        "UCI": 100.0 * float(r["UCI"]) / denom,
        "Disease": r["Disease"],
        "lag_multiplier": r["lag_multiplier"],
    }]))

# Build final plotting DF with only the three plotted metrics
df_plot = pd.concat(derived_rows, ignore_index=True)

# --------------------------------------------------------
# PLOT
# --------------------------------------------------------
save_dir = os.path.join(base_dir, "plots")
os.makedirs(save_dir, exist_ok=True)

years_order = years
scenarios_to_plot = ["S1", "S2", "S3", "S4"]
pollutants = ["PM2.5", "NO2"]

bar_h = 0.16
y_base = np.arange(len(years_order))
scenario_offset = {"S1": -1.5 * bar_h, "S2": -0.5 * bar_h, "S3": +0.5 * bar_h, "S4": +1.5 * bar_h}

whisker_ls = (0, (1.5, 2.2))
whisker_lw = 1.1

def _finite_minmax(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return None, None
    return float(np.min(arr)), float(np.max(arr))

for pollutant in pollutants:
    df_pol = df_plot[df_plot["Pollutant"] == pollutant].copy()
    if df_pol.empty:
        continue

    fig, axes = plt.subplots(
        nrows=3, ncols=1,
        figsize=(8.8, 9.2),
        dpi=220,
        sharey=True,
        gridspec_kw={"height_ratios": [1, 1, 1], "hspace": 0.30},
    )
    fig.suptitle(pollutant, fontsize=14, y=0.98)

    for ax in axes:
        ax.grid(axis="x", linestyle="--", alpha=0.25)
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_yticks(y_base)
        ax.set_yticklabels(years_order)
        ax.tick_params(axis="y", labelleft=True)
        ax.tick_params(axis="x", labelbottom=True)

    def plot_metric(ax, metric, xlabel, alpha=0.85):
        metric_df = df_pol[df_pol["Parameter"] == metric].copy()
        if metric_df.empty:
            ax.set_visible(False)
            return

        metric_vals = metric_df[["Median", "LCI", "UCI"]].apply(pd.to_numeric, errors="coerce")
        xmin, xmax = _finite_minmax(metric_vals[["Median", "LCI"]].to_numpy())
        _, xmax2 = _finite_minmax(metric_vals[["Median", "UCI"]].to_numpy())
        if xmax2 is not None:
            xmax = xmax2 if xmax is None else max(xmax, xmax2)

        if xmin is None or xmax is None:
            ax.set_visible(False)
            return

        xrng = xmax - xmin
        if not np.isfinite(xrng) or xrng <= 0:
            xrng = max(abs(xmax), 1.0)

        for sc in scenarios_to_plot:
            sub = metric_df[metric_df["Scenario"] == sc]
            med, lci, uci = [], [], []
            for yr in years_order:
                row = sub[sub["Year"] == yr]
                if row.empty:
                    med.append(np.nan)
                    lci.append(np.nan)
                    uci.append(np.nan)
                else:
                    med.append(float(row["Median"].iloc[0]))
                    lci.append(float(row["LCI"].iloc[0]))
                    uci.append(float(row["UCI"].iloc[0]))

            med = np.array(med, dtype=float)
            lci = np.array(lci, dtype=float)
            uci = np.array(uci, dtype=float)
            ypos = y_base + scenario_offset[sc]

            ax.barh(
                ypos, med, height=bar_h,
                color=scenario_colors[sc],
                edgecolor="black", linewidth=0.6,
                alpha=alpha,
                zorder=3
            )

            for x0, x1, y in zip(lci, uci, ypos):
                if np.isfinite(x0) and np.isfinite(x1):
                    ax.hlines(
                        y, x0, x1,
                        colors="black",
                        linestyles=whisker_ls,
                        linewidth=whisker_lw,
                        zorder=4
                    )

        left_pad = max(0.08 * xrng, 0.02 * max(abs(xmin), abs(xmax), 1.0))
        right_pad = max(0.10 * xrng, 0.04 * max(abs(xmin), abs(xmax), 1.0))
        ax.set_xlim(min(0, xmin) - left_pad, xmax + right_pad)
        ax.set_ylim(y_base.min() - 0.45 - 1.5 * bar_h, y_base.max() + 0.45 + 1.5 * bar_h)
        ax.invert_yaxis()
        ax.set_xlabel(xlabel)

    plot_metric(axes[0], "MortalityAvoided", "Absolute Premature Mortality Prevented (>30y)")
    plot_metric(
        axes[1],
        "MortalityAvoided_pct_baseline",
        "% of Baseline Pollutant-attributable Mortality Prevented",
        alpha=0.92, 
    )
    plot_metric(
        axes[2],
        "MortalityAvoided_pct_total",
        "% of Total Baseline All-cause Mortality Prevented",
        alpha=0.92
    )

    b = BASELINE_ATTR[pollutant]
    axes[1].set_title(
        f"[Baseline pollutant-attributable mortality (ref to WHO thresholds) = "
        f"{int(b['median']):,} [{int(b['lci']):,}–{int(b['uci']):,}]]",
        fontsize=9.5, pad=4.0
    )
    denom = 583156
    axes[2].set_title(
        f"[Total all-cause mortality: {denom:,}]",
        fontsize=9.5, pad=4.0
    )

    legend_items = [
        Patch(facecolor=scenario_colors["S1"], edgecolor="black", label="S1"),
        Patch(facecolor=scenario_colors["S2"], edgecolor="black", label="S2"),
        Patch(facecolor=scenario_colors["S3"], edgecolor="black", label="S3"),
        Patch(facecolor=scenario_colors["S4"], edgecolor="black", label="S4"),
    ]
    axes[0].legend(handles=legend_items, frameon=True, ncol=5,
                   loc="upper center", bbox_to_anchor=(0.5, 1.22))

    out = os.path.join(save_dir, f"Mortality_3panel_horizontal_LCIUCI_delayed_{pollutant}.png")
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.close(fig)

print(f"Saved plots to: {save_dir}")

Saved plots to: data/2-output-data\plots


In [15]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------------
# Figure 5: Morbidity avoided (cases, YLD) with LCI/UCI as dotted whisker line + end dots
# --------------------------------------------------------
base_dir = "data/2-output-data"

scenarios = ["s1", "s2", "s3", "s4"]
scenario_labels = {"s1": "S1", "s2": "S2", "s3": "S3", "s4": "S4"}
scenario_colors = {"S1": "#9695FF", "S2": "#B47CC7", "S3": "#6ABF69", "S4": "#E88D67"}

pollutant_labels = {
    "ug_PM25_RH50": "PM2.5",
    "ug_NO2": "NO2"
}

years = ["2030", "2050"]

param_titles = {
    "avoided_cases_central": "Total Avoided Cases",
    "YLD_central": "Years Lived with Disability (YLD)",
}

# --------------------------------------------------------
# BUILD DATAFRAME FROM SAVED FILES
# --------------------------------------------------------
records = []

for scenario in scenarios:
    for year in years:
        for pollutant in pollutant_labels.keys():
            filepath = os.path.join(
                base_dir, scenario, pollutant, year, "morbidity_results.csv"
            )

            if not os.path.exists(filepath):
                print(f"⚠ Missing file: {filepath}")
                continue

            df = pd.read_csv(filepath)

            # Group by disease to maintain disease-level granularity
            disease_groups = df.groupby("disease")

            for disease_name, group in disease_groups:
                # --- Morbidity avoided ---
                mort_med = group["avoided_cases_central"].sum()
                mort_lci = group["avoided_cases_low"].sum()
                mort_uci = group["avoided_cases_high"].sum()

                records.append([
                    year, scenario_labels[scenario], pollutant_labels[pollutant],
                    disease_name, "CasesAvoided", mort_med, mort_lci, mort_uci
                ])

                # --- YLD ---
                yld_med = group["YLD_central"].sum()
                yld_lci = group["YLD_low"].sum()
                yld_uci = group["YLD_high"].sum()

                records.append([
                    year, scenario_labels[scenario], pollutant_labels[pollutant],
                    disease_name, "YLD", yld_med, yld_lci, yld_uci
                ])
df_plot_morb = pd.DataFrame(
    records,
    columns=["Year", "Scenario", "Pollutant", "Disease", "Parameter", "Median", "Min", "Max"]
)

# --------------------------
# APPLY LAG MULTIPLIER TO MORTALITY AVOIDED
# --------------------------
# Prepare for merge:
df_plot_morb = df_plot_morb.merge(
    lag_multiplier_df[["Scenario", "Pollutant", "Year", "Disease", "lag_multiplier"]],
    on=["Scenario", "Pollutant", "Year", "Disease"],
    how="left"
)
df_plot_morb["lag_multiplier"] = df_plot_morb["lag_multiplier"].fillna(1.0)
# Apply multiplier to both "avoided_cases_central" and "LE_months"
# For morbidity (e.g. cases, YLD, or your parameter name for morbidity):
for metric in ["avoided_cases_central", "YLD_central"]:  # replace with your morbidity parameter(s) as appropriate
    mask = df_plot_morb["Parameter"] == metric
    for col in ["Median", "Min", "Max"]:
        df_plot_morb.loc[mask, col] = (
                df_plot_morb.loc[mask, col] * df_plot_morb.loc[mask, "lag_multiplier"]
        )
# --------------------------------------------------------
# Figure 5: Bar plots with dotted LCI / UCI for Morbidity (Cases Avoided)
# --------------------------------------------------------
width = 0.22
output_dir = base_dir
os.makedirs(output_dir, exist_ok=True)

morb_titles = {
    "CasesAvoided": "Avoided Cases",
    "YLD": "Years Lived with Disability (YLD)"
}

for pollutant in ["PM2.5", "NO2"]:
    for param in ["CasesAvoided", "YLD"]:

        df_sub = df_plot_morb[
            (df_plot_morb["Pollutant"] == pollutant) &
            (df_plot_morb["Parameter"] == param)
            ].copy()

        if df_sub.empty:
            continue

        # Sort by Year then Disease to ensure 2030 comes first
        df_sub = df_sub.sort_values(["Year", "Disease"])

        # Unique categories for x-axis
        diseases = df_sub["Disease"].unique()
        categories = []
        for yr in years:
            for dis in diseases:
                categories.append(f"{yr}\n{dis}")

        x = np.arange(len(categories))
        fig, ax = plt.subplots(figsize=(14, 7), dpi=300)

        for i, scenario in enumerate(["S1", "S2", "S3", "S4"]):
            # Filter for scenario and align with categories
            medians, lcis, ucis = [], [], []

            for cat in categories:
                yr_val, dis_val = cat.split("\n")
                match = df_sub[
                    (df_sub["Scenario"] == scenario) &
                    (df_sub["Year"] == yr_val) &
                    (df_sub["Disease"] == dis_val)
                    ]

                if not match.empty:
                    medians.append(match["Median"].values[0])
                    lcis.append(match["Min"].values[0])
                    ucis.append(match["Max"].values[0])
                else:
                    medians.append(0)
                    lcis.append(0)
                    ucis.append(0)

            positions = x + (i - 0.5) * width

            # --- Median bars ---
            ax.bar(
                positions,
                medians,
                width,
                color=scenario_colors[scenario],
                edgecolor="black",
                linewidth=0.9,
                alpha=0.95,
                label=scenario
            )

            # --- Dotted LCI–UCI vertical lines ---
            for pos, lo, hi in zip(positions, lcis, ucis):
                ax.vlines(
                    pos,
                    lo,
                    hi,
                    linestyles="dotted",
                    linewidth=1.5,
                    color="black",
                    alpha=0.95
                )

        # Add vertical line to separate 2030 and 2050
        sep_pos = len(diseases) - 0.5
        ax.axvline(x=sep_pos, color='grey', linestyle='--', linewidth=1.5, alpha=0.7)

        # Set Primary X-axis (Diseases)
        ax.set_xticks(x)
        ax.set_xticklabels([cat.split("\n")[1] for cat in categories], fontsize=12, rotation=45, ha="right")

        # Add Secondary X-axis for Years
        for i, yr in enumerate(years):
            center_pos = (len(diseases) - 1) / 2 + i * len(diseases)
            ax.text(center_pos, -0.20, yr, transform=ax.get_xaxis_transform(),
                    ha='center', va='top', fontsize=12, weight='bold')

        ax.set_ylabel(morb_titles.get(param, param), fontsize=12)
        ax.set_title(f"{morb_titles.get(param, param)} by Disease and Year — {pollutant}", fontsize=15, weight="bold")
        ax.grid(axis="y", linestyle="--", alpha=0.5)
        ax.set_axisbelow(True)
        ax.legend(title="Scenario")
        save_dir = "data/2-output-data/plots"
        plt.tight_layout()
        outpath = os.path.join(save_dir, f"morbidity_delayed_{param}_{pollutant}.png")
        plt.savefig(outpath, dpi=300, bbox_inches="tight")
        plt.close(fig)
print("Done — Morbidity bar plots generated with separated years")

Done — Morbidity bar plots generated with separated years


In [16]:
import os
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.patches import Patch

# --------------------------------------------------------
# Simplified DALY Check:
# - YLG (from mortality_chimere_mc.csv): treat as YLL averted (i.e., years of life gained)
# - YLD (from morbidity_results.csv)
# - DALY averted = YLG + YLD
# Output: simple stacked bar plots by scenario/year/pollutant
# --------------------------------------------------------

rcParams["font.family"] = "DejaVu Sans"
warnings.simplefilter(action="ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

base_path = "data/2-output-data"
plot_dir = os.path.join(base_path, "plots")
os.makedirs(plot_dir, exist_ok=True)

scenarios = ["S1", "S2", "S3", "S4"]
pollutants = ["ug_PM25_RH50", "ug_NO2"]
pollutant_label = {"ug_PM25_RH50": "PM2.5", "ug_NO2": "NO2"}
years = ["2030", "2050"]

scenario_colors = {"S1": "#9695FF", "S2": "#B47CC7", "S3": "#6ABF69", "S4": "#E88D67"}
yll_color = "#2E4057"
yld_color = "#F28E2B"
MORT_DISEASE_LABEL = "All-cause mortality (main)"
MORB_DISEASE_LABEL = "TOTAL morbidity (excl. mortality)"

# --- Get a lookup for each multiplier ---
lag_df = pd.read_csv(INFILE)
def get_multiplier(scenario, pollutant, year, disease):
    # This merges on all 4 keys; in real use, you may want a dict lookup for speed.
    row = lag_df[
        (lag_df["Scenario"] == scenario)
        & (lag_df["Pollutant"] == pollutant)
        & (lag_df["Year"] == str(year))
        & (lag_df["Disease"] == disease)
    ]
    if not row.empty:
        return float(row.iloc[0]["lag_multiplier"])
    return 1.0
# --------------------------------------------------------
# Build DALY totals table with multiplier
# --------------------------------------------------------
rows = []
for sc in scenarios:
    for pol in pollutants:
        for yr in years:
            mpath = mortality_csv_path(sc, pol, yr)
            bpath = morbidity_csv_path(sc, pol, yr)

            if mpath is None or bpath is None:
                logging.warning(f"Missing file(s) for {sc}/{pol}/{yr}: mortality={mpath} morbidity={bpath}")
                continue

            ylg_med, ylg_lci, ylg_uci = load_total_ylg(mpath)
            yld_med, yld_lci, yld_uci = load_total_yld(bpath)

            # Get multipliers
            mort_mult = get_multiplier(sc, pol, yr, MORT_DISEASE_LABEL)
            morb_mult = get_multiplier(sc, pol, yr, MORB_DISEASE_LABEL)

            # Apply multipliers
            ylg_med *= mort_mult
            ylg_lci *= mort_mult if np.isfinite(ylg_lci) else np.nan
            ylg_uci *= mort_mult if np.isfinite(ylg_uci) else np.nan

            yld_med *= morb_mult
            yld_lci *= morb_mult if np.isfinite(yld_lci) else np.nan
            yld_uci *= morb_mult if np.isfinite(yld_uci) else np.nan

            # DALY averted (simple sum)
            daly_med = ylg_med + yld_med
            daly_lci = (ylg_lci + yld_lci) if np.isfinite(ylg_lci) and np.isfinite(yld_lci) else np.nan
            daly_uci = (ylg_uci + yld_uci) if np.isfinite(ylg_uci) and np.isfinite(yld_uci) else np.nan

            rows.append({
                "Scenario": sc,
                "Pollutant": pol,
                "Year": str(yr),
                "YLG_med": ylg_med, "YLG_lci": ylg_lci, "YLG_uci": ylg_uci,
                "YLD_med": yld_med, "YLD_lci": yld_lci, "YLD_uci": yld_uci,
                "DALY_med": daly_med, "DALY_lci": daly_lci, "DALY_uci": daly_uci,
            })

daly_df = pd.DataFrame(rows)
if daly_df.empty:
    raise SystemExit("No DALY rows computed (check paths / columns).")
###########################
#### PLOT CODE ######
##########################
bar_w = 0.18
whisker_ls = (0, (1.5, 2.2))
whisker_lw = 1.1
dot_ms = 4.0
years_order = [str(y) for y in years]
x_base = np.arange(len(years_order), dtype=float)

def _scenario_sort_key(s):
    if isinstance(s, str) and len(s) >= 2 and s[0].upper() == "S" and s[1:].isdigit():
        return int(s[1:])
    return 999

scenarios_order = sorted(daly_df["Scenario"].unique().tolist(), key=_scenario_sort_key)
if len(scenarios_order) == 1:
    offsets = {scenarios_order[0]: 0.0}
else:
    spread = min(0.70, max(0.28, len(scenarios_order) * bar_w + 0.10))
    offs = np.linspace(-spread / 2, spread / 2, len(scenarios_order))
    offsets = {sc: float(o) for sc, o in zip(scenarios_order, offs)}

for pol in sorted(daly_df["Pollutant"].unique().tolist()):
    dfp = daly_df[daly_df["Pollutant"] == pol].copy()
    if dfp.empty:
        continue

    fig, ax = plt.subplots(figsize=(9.3, 4.9), dpi=240)
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    ax.grid(axis='x', visible=False)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.set_xticks(x_base)
    ax.set_xticklabels(years_order, weight="bold")
    ax.set_xlabel("Year", weight="bold")
    ax.set_ylabel("DALYs averted (YLG + YLD)", weight="bold")
    ax.set_title(f"DALYs averted — {pollutant_label.get(pol, pol)}", weight="bold", pad=10)

    for sc in scenarios_order:
        sub = dfp[dfp["Scenario"] == sc]

        ylg = []
        yld = []
        daly_lci = []
        daly_uci = []
        for yr in years_order:
            row = sub[sub["Year"] == yr]
            if row.empty:
                ylg.append(np.nan)
                yld.append(np.nan)
                daly_lci.append(np.nan)
                daly_uci.append(np.nan)
            else:
                ylg.append(float(row["YLG_med"].iloc[0]))
                yld.append(float(row["YLD_med"].iloc[0]))
                daly_lci.append(float(row["DALY_lci"].iloc[0]) if np.isfinite(row["DALY_lci"].iloc[0]) else np.nan)
                daly_uci.append(float(row["DALY_uci"].iloc[0]) if np.isfinite(row["DALY_uci"].iloc[0]) else np.nan)

        ylg = np.array(ylg, dtype=float)
        yld = np.array(yld, dtype=float)
        daly_lci = np.array(daly_lci, dtype=float)
        daly_uci = np.array(daly_uci, dtype=float)

        xpos = x_base + offsets[sc]
        # stack: YLD (bottom) + YLG (top)
        ax.bar(
            xpos, yld, width=bar_w,
            color=yld_color, edgecolor="black", linewidth=0.6,
            alpha=0.6, zorder=3
        )
        ax.bar(
            xpos, ylg, width=bar_w, bottom=yld,
            color=yll_color, edgecolor="black", linewidth=0.6,
            alpha=0.6, zorder=3
        )
        # scenario outline cue (optional): thin colored edge around full stack
        # (keeps colors for components but still distinguishes scenarios)
        total = yld + ylg
        ax.bar(
            xpos, total, width=bar_w,
            facecolor="none",
            edgecolor=scenario_colors.get(sc, "#333333"),
            linewidth=1.2,
            zorder=4,
            label=sc
        )
        for x, lo, hi in zip(xpos, daly_lci, daly_uci):
            if np.isfinite(lo) and np.isfinite(hi):
                ax.vlines(x, lo, hi, colors="black", linestyles=whisker_ls,
                          linewidth=whisker_lw, zorder=5)
                ax.plot([x, x], [lo, hi], linestyle="none", marker="o",
                        markersize=dot_ms, color="black", zorder=6)

    handles = [
        Patch(facecolor=yld_color, edgecolor="black", label="YLD (morbidity proportion)"),
        Patch(facecolor=yll_color, edgecolor="black", label="YLG (mortality proportion)"),
        Patch(facecolor="white", edgecolor=scenario_colors["S1"], label="S1 outline"),
        Patch(facecolor="white", edgecolor=scenario_colors["S2"], label="S2 outline"),
        Patch(facecolor="white", edgecolor=scenario_colors["S3"], label="S3 outline"),
        Patch(facecolor="white", edgecolor=scenario_colors["S4"], label="S4 outline"),
    ]
    ax.legend(handles=handles, frameon=True, ncol=2, loc="upper left", fontsize=10)
    out_file = os.path.join(plot_dir, f"DALY_DELAYED_{pol}.png")
    fig.savefig(out_file, dpi=300, bbox_inches="tight")
    logging.info(f"Saved figure: {out_file}")
    plt.close(fig)

2026-03-31 17:23:13,235 - INFO - Saved figure: data/2-output-data\plots\DALY_DELAYED_ug_NO2.png
2026-03-31 17:23:13,437 - INFO - Saved figure: data/2-output-data\plots\DALY_DELAYED_ug_PM25_RH50.png


In [20]:
import os
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.patches import Patch

rcParams["font.family"] = "DejaVu Sans"
warnings.simplefilter(action="ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

base_path = "data/2-output-data"
plot_dir = os.path.join(base_path, "plots")
os.makedirs(plot_dir, exist_ok=True)

scenarios = ["S1", "S2", "S3", "S4"]
pollutants = ["ug_PM25_RH50", "ug_NO2"]
poll_map = {
    "PM2.5": "ug_PM25_RH50",
    "PM25": "ug_PM25_RH50",
    "ug_PM25_RH50": "ug_PM25_RH50",
    "NO2": "ug_NO2",
    "ug_NO2": "ug_NO2",
}
pollutant_label = {"ug_PM25_RH50": "PM2.5", "ug_NO2": "NO2"}
years = ["2030", "2050"]

scenario_colors = {"S1": "#9695FF", "S2": "#B47CC7", "S3": "#6ABF69", "S4": "#E88D67"}
yll_color = "#2E4057"
yld_color = "#F28E2B"

bar_w = 0.18
years_order = [str(y) for y in years]
x_base = np.arange(len(years_order), dtype=float)

# --------------------------------------------------------
# LOAD DALY DATA
# --------------------------------------------------------
rows = []
for sc in scenarios:
    for pol in pollutants:
        for yr in years:
            mpath = mortality_csv_path(sc, pol, yr)
            bpath = morbidity_csv_path(sc, pol, yr)

            if mpath is None or bpath is None:
                continue

            ylg_med, _, _ = load_total_ylg(mpath)
            yld_med, _, _ = load_total_yld(bpath)

            rows.append({
                "Scenario": sc,
                "Pollutant": pol,
                "Year": str(yr),
                "YLG": ylg_med,
                "YLD": yld_med,
            })

daly_df = pd.DataFrame(rows)

# --------------------------------------------------------
# LOAD INTANGIBLE COST DATA
# --------------------------------------------------------
INFILE = os.path.join("data", "2-output-data", "economic_summary_morbidity_sensitivity.csv")
df = pd.read_csv(INFILE)

df = df[df["Lag"].astype(str).str.lower() == "cessation_lag"].copy()

# IMPORTANT FIX: force alignment
df["Scenario"] = df["Scenario"].apply(clean_scenario).str.upper()
df["Pollutant"] = df["Pollutant"].astype(str).str.strip().map(poll_map).fillna(df["Pollutant"].astype(str).str.strip())
df["Year"] = df["Year"].astype(str)


# numeric conversion (robust)
def _safe_float(x):
    return pd.to_numeric(x, errors="coerce")


df["IntYLL_c_MEUR"] = np.where(
    df["Disease"].astype(str).str.strip().eq("All-cause mortality (main)"),
    df["Intangible Cost YLL (M€)"].apply(_safe_float),
    np.nan
)
df["IntYLD_c_MEUR"] = df["Intangible Cost YLD (M€)"].apply(_safe_float)

# aggregate
int_df = (
    df.groupby(["Scenario", "Pollutant", "Year"], as_index=False)[
        ["IntYLL_c_MEUR", "IntYLD_c_MEUR"]
    ]
    .sum(min_count=1)
)

# DEBUG (optional but very useful)
logging.info(f"Intangible scenarios present: {int_df['Scenario'].unique()}")


# --------------------------------------------------------
# Scenario offsets
# --------------------------------------------------------
def _scenario_sort_key(s):
    return int(s[1:]) if s[1:].isdigit() else 999


scenarios_order = sorted(scenarios, key=_scenario_sort_key)
offs = np.linspace(-0.35, 0.35, len(scenarios_order))
offsets = dict(zip(scenarios_order, offs))

# --------------------------------------------------------
# PLOTTING
# --------------------------------------------------------
for pol in pollutants:

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=240, sharey=False)

    for ax, mode in zip(axes, ["DALY", "INT"]):

        ax.grid(axis="y", linestyle="--", alpha=0.25)
        ax.grid(axis='x', visible=False)
        ax.set_axisbelow(True)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        ax.set_xticks(x_base)
        ax.set_xticklabels(years_order, weight="bold")
        ax.set_xlabel("Year", weight="bold")
        ax.set_ylabel("Value", weight="bold")

        if mode == "DALY":
            ax.set_title(f"DALY — {pollutant_label[pol]}", weight="bold")
        else:
            ax.set_title(f"Intangible cost — {pollutant_label[pol]}", weight="bold")

        for sc in scenarios_order:

            yll_vals = []
            yld_vals = []

            for yr in years_order:

                if mode == "DALY":
                    row = daly_df[
                        (daly_df["Scenario"] == sc) &
                        (daly_df["Pollutant"] == pol) &
                        (daly_df["Year"] == yr)
                        ]

                    if row.empty:
                        yll_vals.append(np.nan)
                        yld_vals.append(np.nan)
                    else:
                        yll_vals.append(row["YLG"].iloc[0])
                        yld_vals.append(row["YLD"].iloc[0])

                else:
                    row = int_df[
                        (int_df["Scenario"] == sc) &
                        (int_df["Pollutant"] == pol) &
                        (int_df["Year"] == yr)
                        ]

                    if row.empty:
                        logging.warning(f"No INT data for {sc}-{pol}-{yr}")
                        yll_vals.append(np.nan)
                        yld_vals.append(np.nan)
                    else:
                        yll_vals.append(row["IntYLL_c_MEUR"].iloc[0])
                        yld_vals.append(row["IntYLD_c_MEUR"].iloc[0])

            yll_vals = np.array(yll_vals, dtype=float)
            yld_vals = np.array(yld_vals, dtype=float)

            xpos = x_base + offsets[sc]

            ax.bar(xpos, yld_vals, width=bar_w,
                   color=yld_color, edgecolor="black", linewidth=0.6, alpha=0.8)

            ax.bar(xpos, yll_vals, width=bar_w, bottom=yld_vals,
                   color=yll_color, edgecolor="black", linewidth=0.6, alpha=0.8)

            ax.bar(xpos, yll_vals + yld_vals, width=bar_w,
                   facecolor="none",
                   edgecolor=scenario_colors[sc],
                   linewidth=3.0,
                   zorder=4
                   )

    handles = [
        Patch(facecolor=yld_color, edgecolor="black", label="YLD"),
        Patch(facecolor=yll_color, edgecolor="black", label="YLG"),
        Patch(facecolor="none", edgecolor=scenario_colors["S1"], label="S1"),
        Patch(facecolor="none", edgecolor=scenario_colors["S2"], label="S2"),
        Patch(facecolor="none", edgecolor=scenario_colors["S3"], label="S3"),
        Patch(facecolor="none", edgecolor=scenario_colors["S4"], label="S4"),
    ]
    axes[0].legend(handles=handles, frameon=True, ncol=6, fontsize=10, loc="upper center", bbox_to_anchor=(0.9, -0.15))
    out_file = os.path.join(plot_dir, f"DALY_vs_Intangible_absolute_{pol}.png")
    fig.savefig(out_file, dpi=300, bbox_inches="tight")
    logging.info(f"Saved figure: {out_file}")
    plt.close(fig)

2026-03-31 17:36:35,014 - INFO - Intangible scenarios present: ['S1' 'S2' 'S3' 'S4']
2026-03-31 17:36:35,364 - INFO - Saved figure: data/2-output-data\plots\DALY_vs_Intangible_absolute_ug_PM25_RH50.png
2026-03-31 17:36:36,110 - INFO - Saved figure: data/2-output-data\plots\DALY_vs_Intangible_absolute_ug_NO2.png


**Supplementary Figures**

In [16]:
import os
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

rcParams["font.family"] = "DejaVu Sans"
warnings.simplefilter(action="ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# --------------------------------------------------------
# SETTINGS
# --------------------------------------------------------
base_path = "data/2-output-data"
plot_dir = os.path.join(base_path, "plots")
os.makedirs(plot_dir, exist_ok=True)

scenarios = ["S1", "S2", "S3", "S4"]
pollutants = ["ug_PM25_RH50", "ug_NO2"]
years = ["2030", "2050"]

pollutant_title = {"ug_PM25_RH50": "PM2.5", "ug_NO2": "NO2"}

scenario_colors = {"S1": "#9695FF", "S2": "#B47CC7", "S3": "#6ABF69", "S4": "#E88D67"}

age_bins = [-np.inf, 9, 19, 29, 39, 49, 59, 69, 79, 89, np.inf]
age_labels = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80-89", "90-99"]

disease_age_groups = {
    "Hypertension": (18, 99),
    "Lung Cancer": (35, 99),
    "Asthma in children": (0, 17),
    "Asthma in adult": (18, 99),
    "COPD": (40, 99),
    "ALRI in children": (0, 12),
    "Stroke": (35, 99),
    "IHD events": (30, 99),
    "Diabetes T2": (45, 99),
}

colors = ["#F28E2B", "#E15759", "#76B7B2", "#59A14F", "#EDC948", "#B07AA1", "#FF9DA7", "#D4C23D", "#4E79A7"]
ylg_color = "#2E4057"  # base (YLG / YLL averted proxy)
# --------------------------------------------------------
# MULTIPLIERS
# --------------------------------------------------------
lag_df = pd.read_csv(INFILE)
# --------------------------------------------------------
# LOAD MORBIDITY
# --------------------------------------------------------
morb_list = []
for pol in pollutants:
    for sc in scenarios:
        for yr in years:
            p = morbidity_csv_path(sc, pol, yr)
            if p:
                d = norm_cols(pd.read_csv(p))
                d["scenario"], d["pollutant"], d["year"] = sc, pol, str(yr)
                morb_list.append(d)

morb = pd.concat(morb_list, ignore_index=True)

age_col = pick_col(morb, ["age"])
disease_col = pick_col(morb, ["disease"])
yld_col = pick_col(morb, ["YLD_central", "YLD"])

morb = morb.rename(columns={age_col: "age", disease_col: "disease"})
morb["age"] = pd.to_numeric(morb["age"], errors="coerce")
morb[yld_col] = pd.to_numeric(morb[yld_col], errors="coerce").fillna(0)

morb = filter_morbidity_diseases(morb)
morb["age_group"] = pd.cut(morb["age"], bins=age_bins, labels=age_labels)

# APPLY MORBIDITY MULTIPLIER (per disease row)
morb["multiplier"] = morb.apply(
    lambda r: get_multiplier(r["scenario"], r["pollutant"], r["year"], r["disease"]),
    axis=1
)
morb[yld_col] *= morb["multiplier"]

morb_agg = (
    morb.groupby(["pollutant", "year", "scenario", "age_group", "disease"])[yld_col]
    .sum().reset_index().rename(columns={yld_col: "YLD_plot"})
)

# --------------------------------------------------------
# LOAD MORTALITY
# --------------------------------------------------------
mort_list = []
for pol in pollutants:
    for sc in scenarios:
        for yr in years:
            p = mortality_csv_path(sc, pol, yr)
            if p:
                d = norm_cols(pd.read_csv(p))
                d["scenario"], d["pollutant"], d["year"] = sc, pol, str(yr)
                mort_list.append(d)

mort = pd.concat(mort_list, ignore_index=True)

ylg_col = pick_col(mort, ["YLG"])
age_col_m = pick_col(mort, ["age"])

mort["YLG_plot"] = pd.to_numeric(mort[ylg_col], errors="coerce").fillna(0)
mort["age"] = pd.to_numeric(mort[age_col_m], errors="coerce")
mort["age_group"] = pd.cut(mort["age"], bins=age_bins, labels=age_labels)

# APPLY MORTALITY MULTIPLIER (single label)
MORT_LABEL = "Mortality"

mort["multiplier"] = mort.apply(
    lambda r: get_multiplier(r["scenario"], r["pollutant"], r["year"], MORT_LABEL),
    axis=1
)
mort["YLG_plot"] *= mort["multiplier"]

mort_agg = (
    mort.groupby(["pollutant", "year", "scenario", "age_group"])["YLG_plot"]
    .sum().reset_index()
)
# --------------------------------------------------------
# MERGE
# --------------------------------------------------------
agg = morb_agg.merge(mort_agg, on=["pollutant", "year", "scenario", "age_group"], how="left")
agg["YLG_plot"] = agg["YLG_plot"].fillna(0)

yld_pivot = agg.pivot_table(
    index=["pollutant", "year", "scenario", "age_group"],
    columns="disease",
    values="YLD_plot",
    fill_value=0
).reset_index()

plot_df = yld_pivot.merge(mort_agg, on=["pollutant", "year", "scenario", "age_group"], how="left")

# Fill only numeric columns to avoid categorical fillna TypeError
numeric_cols = plot_df.select_dtypes(include=[np.number]).columns
plot_df[numeric_cols] = plot_df[numeric_cols].fillna(0)

# disease ordering + colors (FIXED)
disease_list = [d for d in disease_age_groups.keys() if d in plot_df.columns]
disease_color_map = {d: colors[i % len(colors)] for i, d in enumerate(disease_list)}

# --------------------------------------------------------
# OVERALL
# --------------------------------------------------------
plot_df["YLD_total"] = plot_df[disease_list].sum(axis=1)
plot_df["DALY_total"] = plot_df["YLD_total"] + plot_df["YLG_plot"]
overall_summary = (
    plot_df.groupby(["pollutant", "year", "scenario"], as_index=False)["DALY_total"]
    .sum()
)
# --------------------------------------------------------
# PLOT
# --------------------------------------------------------
for pol in plot_df["pollutant"].unique():

    sub_pol = plot_df[plot_df["pollutant"] == pol]

    fig, axes = plt.subplots(len(years), len(scenarios) + 1,
                             figsize=(4 * (len(scenarios) + 1), 5 * len(years)),
                             sharey=False)

    axes = np.array(axes).reshape(len(years), len(scenarios) + 1)

    age_panel_max = 0
    for yr in years:
        year_block = sub_pol[sub_pol["year"] == yr]
        if not year_block.empty:
            year_vals = year_block[disease_list + ["YLG_plot"]].sum(axis=1).max()
            age_panel_max = max(age_panel_max, float(year_vals) if pd.notna(year_vals) else 0)
    age_panel_max = age_panel_max * 1.15 if age_panel_max > 0 else 1

    for i, yr in enumerate(years):

        # ---- AGE PANELS ----
        for j, sc in enumerate(scenarios):
            ax = axes[i, j]

            block = sub_pol[(sub_pol["year"] == yr) & (sub_pol["scenario"] == sc)]
            block = block.sort_values("age_group")

            x = np.arange(len(age_labels))
            ylg_vals = block["YLG_plot"].values

            ax.bar(x, ylg_vals, color=ylg_color, edgecolor="black", linewidth=0.4)

            bottom = ylg_vals.copy()
            for d in disease_list:
                vals = block[d].values
                ax.bar(x, vals, bottom=bottom,
                       color=disease_color_map[d])
                bottom += vals

            ax.set_title(f"{sc} — {yr}")
            ax.set_xticks(x)
            ax.set_xticklabels(age_labels, rotation=40)
            ax.grid(axis="y", linestyle="--", alpha=0.3)
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
            ax.set_ylim(0, age_panel_max)

        # ---- OVERALL PANEL (OWN SCALE) ----
        ax = axes[i, -1]

        block = (
            overall_summary[
                (overall_summary["pollutant"] == pol) &
                (overall_summary["year"] == yr)
                ]
            .set_index("scenario")
            .reindex(scenarios)
        )

        x = np.arange(len(scenarios))
        vals = block["DALY_total"].values

        ax.bar(x, vals,
               color=[scenario_colors[s] for s in scenarios],
               edgecolor="black", linewidth=0.5)

        ax.set_title(f"Overall — {yr}")
        ax.set_xticks(x)
        ax.set_xticklabels(scenarios)

        # KEY: independent y-axis, but consistent across 2030 and 2050 for each pollutant
        pol_vals = overall_summary[overall_summary["pollutant"] == pol]["DALY_total"].values
        max_ylim = max(pol_vals) * 1.15 if len(pol_vals) else 1
        ax.set_ylim(0, max_ylim)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.grid(axis="y", linestyle="--", alpha=0.3)

    # ---- LEGEND ----
    available_diseases = [d for d in disease_list if sub_pol[d].to_numpy().sum() > 0]
    legend_handles = [Patch(facecolor=ylg_color, label="YLL")]
    legend_handles += [Patch(facecolor=disease_color_map[d], label=d) for d in available_diseases]
    fig.legend(handles=legend_handles, loc="lower center",
               ncol=min(8, max(1, len(legend_handles))), bbox_to_anchor=(0.5, 0.01))

    fig.suptitle(f"DALYs averted — {pollutant_title.get(pol, pol)}",
                 fontsize=16, weight="bold")

    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.savefig(os.path.join(plot_dir, f"DALY_with_overall_{pol}.png"), dpi=300)
    logging.info(f"Saved figure: {plot_dir}")
    plt.close()

2026-03-30 12:27:43,470 - INFO - Saved figure: data/2-output-data\plots
2026-03-30 12:27:45,524 - INFO - Saved figure: data/2-output-data\plots


In [6]:
#---------------------------------------------------
#Supplementary Figure 5 and 6: Heatmap by scenario for each pollutant and year
#---------------------------------------------------
"""
Heatmaps:
- Per year (2030, 2050) and combined pollutants (PM2.5 + NO2), plot 3 heatmaps (outcomes x scenarios):
  1) Events prevented (Cases for morbidity rows; plus a "TOTAL morbidity" row)
  2) Direct medical costs prevented (M€) (morbidity + a "TOTAL morbidity" row)
  3) Intangible costs prevented (M€) where Intangible = YLD (morbidity + total morbidity)
"""

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------
# Config
# -------------------------
INFILE = os.path.join("data", "2-output-data", "economic_summary_morbidity_sensitivity.csv")
OUTDIR = os.path.join("data", "2-output-data", "plots")
os.makedirs(OUTDIR, exist_ok=True)

SCEN_ORDER = ["s1", "s2", "s3", "s4"]
POLL_ORDER = ["PM2.5", "NO2"]
YEAR_ORDER = ["2030", "2050"]

TOTAL_MORB_LABEL = "TOTAL morbidity (excl. mortality)"

# Outcomes to display individually (optional).
# We will add TOTAL_MORB_LABEL automatically.
OUTCOME_ORDER = [
    "Hypertension",
    "Diabetes T2",
    "COPD",
    "Stroke",
    "IHD events",
    "AMI events",
    "Lung Cancer",
    "Asthma in children",
    "Asthma in adult",
    "ALRI in children",
    TOTAL_MORB_LABEL,
]

OUTCOME_LABELS = {
    "Diabetes T2": "Type 2 diabetes",
    "Asthma in adult": "Asthma in adults",
    TOTAL_MORB_LABEL: "TOTAL Morbidity",
}

# -------------------------
# Load and preprocess
# -------------------------
df = pd.read_csv(INFILE)

# Filter no_lag only
df = df[df["Lag"].astype(str).str.lower() == "cessation_lag"].copy()
if df.empty:
    raise RuntimeError("No rows found with Lag == 'cessation_lag'. Check your CSV.")

df["Scenario"] = df["Scenario"].apply(clean_scenario)
df["Year"] = df["Year"].astype(str)
df["Pollutant"] = df["Pollutant"].astype(str)
df["Disease"] = df["Disease"].astype(str)

# Parse cases/deaths strings into numeric
cases = df["Avoided Cases"].apply(parse_est_ci)
df[["Cases_c", "Cases_l", "Cases_u"]] = pd.DataFrame(cases.tolist(), index=df.index)

deaths = df["Deaths Avoided"].apply(parse_est_ci)
df[["Deaths_c", "Deaths_l", "Deaths_u"]] = pd.DataFrame(deaths.tolist(), index=df.index)

# Costs: numeric in M€ columns; coerce blank -> NaN
df["DirectMed_c_MEUR"] = df["Direct Med Cost (M€)"].apply(_to_float)
df["DirectMed_l_MEUR"] = df["Direct Med Cost LCI (M€)"].apply(_to_float)
df["DirectMed_u_MEUR"] = df["Direct Med Cost UCI (M€)"].apply(_to_float)

df["IntYLL_c_MEUR"] = df["Intangible Cost YLL (M€)"].apply(_to_float)
df["IntYLL_l_MEUR"] = df["Intangible Cost YLL LCI (M€)"].apply(_to_float)
df["IntYLL_u_MEUR"] = df["Intangible Cost YLL UCI (M€)"].apply(_to_float)

df["IntYLD_c_MEUR"] = df["Intangible Cost YLD (M€)"].apply(_to_float)
df["IntYLD_l_MEUR"] = df["Intangible Cost YLD LCI (M€)"].apply(_to_float)
df["IntYLD_u_MEUR"] = df["Intangible Cost YLD UCI (M€)"].apply(_to_float)

# Combined intangible = YLL + YLD (treat NaN as 0 for the sum)
for b in ["c", "l", "u"]:
    df[f"IntTotal_{b}_MEUR"] = df[f"IntYLL_{b}_MEUR"].fillna(0) + df[f"IntYLD_{b}_MEUR"].fillna(0)

# Keep only scenarios/pollutants/years we want
df = df[df["Scenario"].isin(SCEN_ORDER) & df["Pollutant"].isin(POLL_ORDER) & df["Year"].isin(YEAR_ORDER)].copy()

# -------------------------
# Build "TOTAL morbidity" rows (per scenario/pollutant/year)
# -------------------------
morb = df[~df["Disease"].apply(is_any_mortality_variant)].copy()

tot_rows = []
group_cols = ["Scenario", "Pollutant", "Year"]
for (sc, pol, yr), g in morb.groupby(group_cols, dropna=False):
    tot_rows.append({
        "Scenario": sc,
        "Pollutant": pol,
        "Year": yr,
        "Disease": TOTAL_MORB_LABEL,
        "Cases_c": safe_sum(g["Cases_c"]),
        "Cases_l": safe_sum(g["Cases_l"]),
        "Cases_u": safe_sum(g["Cases_u"]),
        "Deaths_c": np.nan,
        "Deaths_l": np.nan,
        "Deaths_u": np.nan,
        "DirectMed_c_MEUR": safe_sum(g["DirectMed_c_MEUR"]),
        "DirectMed_l_MEUR": safe_sum(g["DirectMed_l_MEUR"]),
        "DirectMed_u_MEUR": safe_sum(g["DirectMed_u_MEUR"]),
        "IntTotal_c_MEUR": safe_sum(g["IntTotal_c_MEUR"]),
        "IntTotal_l_MEUR": safe_sum(g["IntTotal_l_MEUR"]),
        "IntTotal_u_MEUR": safe_sum(g["IntTotal_u_MEUR"]),
        # keep these for consistency (not used directly)
        "IntYLL_c_MEUR": np.nan,
        "IntYLL_l_MEUR": np.nan,
        "IntYLL_u_MEUR": np.nan,
        "IntYLD_c_MEUR": safe_sum(g["IntYLD_c_MEUR"]),
        "IntYLD_l_MEUR": safe_sum(g["IntYLD_l_MEUR"]),
        "IntYLD_u_MEUR": safe_sum(g["IntYLD_u_MEUR"]),
    })

tot_df = pd.DataFrame(tot_rows)

# For plotting, merge original + totals and keep only outcomes we care about
plot_df = pd.concat([df, tot_df], ignore_index=True)
plot_df["OutcomeLabel"] = plot_df["Disease"].apply(outcome_display_name)


# If both "Diabetes T2" and "Type 2 diabetes" appear, you might get duplicates.
# We'll keep whichever exists; pivot_table with aggfunc='max' resolves duplicates safely.
def _dedup_keep_max(pv):
    return pv.groupby(level=0).max()


# -------------------------
# Plot heatmaps
# -------------------------
sns.set_theme(style="whitegrid")


def pivot_metric(dd, value_col):
    pv = dd.pivot_table(
        index="OutcomeLabel",
        columns="Scenario",
        values=value_col,
        aggfunc="max",  # robust to duplicates
    )
    # order rows and cols
    ordered_rows = [outcome_display_name(o) for o in OUTCOME_ORDER if outcome_display_name(o) in pv.index]
    pv = pv.reindex(ordered_rows)
    pv = pv.reindex(columns=SCEN_ORDER)
    return pv


def make_heatmaps_for_year(year):
    d = plot_df[plot_df["Year"] == str(year)].copy()
    if d.empty:
        print(f"[WARN] No data for year {year}")
        return

    # Combine PM2.5 + NO2 together for a single heatmap set per year.
    # Missing NO2 diseases are handled naturally by treating absent rows as NaN and summing only available values.
    dd = d.groupby(["Scenario", "Disease"], as_index=False).agg({
        "Cases_c": lambda s: s.sum(min_count=1),
        "DirectMed_c_MEUR": lambda s: s.sum(min_count=1),
        "IntYLD_c_MEUR": lambda s: s.sum(min_count=1),
    }).copy()

    # Re-attach labels and keep only the desired outcome order
    dd["OutcomeLabel"] = dd["Disease"].apply(outcome_display_name)

    pv_events = pivot_metric(dd, "Cases_c")
    pv_tang = pivot_metric(dd, "DirectMed_c_MEUR")
    pv_intang = pivot_metric(dd, "IntYLD_c_MEUR")

    fig, axes = plt.subplots(1, 3, figsize=(20.5, 7))
    fig.suptitle(f" Morbidity and Economic Co-benefits: PM2.5 + NO2 — {year}", fontsize=14, y=1.02, fontweight="bold")
    fig.subplots_adjust(
        top=0.9,
        bottom=0.05,
        left=0.05,
        right=0.965,
        wspace=0.42,
        hspace=0.3,
    )

    # Heatmap 1: events
    sns.heatmap(
        pv_events,
        ax=axes[0],
        cmap="YlGnBu",
        linewidths=0.4,
        linecolor="white",
        annot=True,
        fmt=".0f",
        cbar_kws={"label": "Prevented cases", "pad": 0.03, "shrink": 0.92},
    )
    axes[0].set_title("Morbidity cases prevented")
    axes[0].set_xlabel("Scenario")
    axes[0].set_ylabel("Outcome")

    # Heatmap 2: tangible costs (M€)
    sns.heatmap(
        pv_tang,
        ax=axes[1],
        cmap="YlGnBu",
        linewidths=0.4,
        linecolor="white",
        annot=True,
        fmt=".0f",
        cbar_kws={"label": "Direct medical costs prevented (M€)", "pad": 0.03, "shrink": 0.92},
    )
    axes[1].set_title("Direct medical costs prevented")
    axes[1].set_xlabel("Scenario")
    axes[1].set_ylabel("")

    # Heatmap 3: intangible costs (YLD)
    sns.heatmap(
        pv_intang,
        ax=axes[2],
        cmap="YlGnBu",
        linewidths=0.4,
        linecolor="white",
        annot=True,
        fmt=".0f",
        cbar_kws={"label": "Intangible costs prevented (M€, YLD)", "pad": 0.03, "shrink": 0.92},
    )
    axes[2].set_title("Intangible costs prevented")
    axes[2].set_xlabel("Scenario")
    axes[2].set_ylabel("")

    out_png = os.path.join(OUTDIR, f"heatmaps_combined_{year}_delayed.png")
    fig.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved heatmaps: {out_png}")


# -------------------------
# Run
# -------------------------
for y in YEAR_ORDER:
    make_heatmaps_for_year(y)
print(f"\nDone. Outputs saved in: {OUTDIR}")

[OK] Saved heatmaps: data\2-output-data\plots\heatmaps_combined_2030_delayed.png
[OK] Saved heatmaps: data\2-output-data\plots\heatmaps_combined_2050_delayed.png

Done. Outputs saved in: data\2-output-data\plots
